In [3]:
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 계산
import re  # 정규표현식
import time  # 시간 측정

IN_PATH_SUFUL = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사재고수불부모음_ver6_수정.xlsx"  # 수불부 경로
IN_PATH_SUBMIT = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사제출용.xlsx"  # 제출용 경로
OUT_PATH = r"./output_조정본.xlsx"  # 출력 경로
OUT_PATH_SAMPLE = r"./output_조정본_샘플.xlsx"  # 샘플 출력 경로

WAREHOUSE_KEYWORD = "본사"  # 본사만
MAX_LOOP = 200  # 반복 제한
PRINT_EVERY_N_MOVES = 20  # 이동 출력 빈도

t0 = time.time()  # 시작시간

print("1) 파일 로딩 시작")  # 진행 출력
df = pd.read_excel(IN_PATH_SUFUL, sheet_name="Sheet1")  # 수불부 읽기
submit = pd.read_excel(IN_PATH_SUBMIT, sheet_name="본사제출용")  # 제출용 읽기
print("1) 파일 로딩 완료")  # 진행 출력

df = df.copy()  # 원본 보호
submit = submit.copy()  # 원본 보호

1) 파일 로딩 시작
1) 파일 로딩 완료


In [ ]:
SAMPLE_MODE = True  # 샘플 처리 모드
SAMPLE_NEGATIVE_LIMIT = 5  # 샘플용 음수 품목 탐색 수
SAMPLE_MAX_MOVES = 50  # 샘플용 최대 이동 건수
SAMPLE_MAX_LOOPS = 10  # 샘플용 최대 loop
SAVE_SAMPLE_ONLY = True  # 샘플만 저장할지 여부
SAMPLE_ROWS = 5000  # 샘플 행 수

print("2) 일자 파싱 시작")  # 진행 출력
df["일자"] = df["일자"].astype(str).str.strip()  # 일자 문자열 정리
df["일자"] = df["일자"].str.replace(r"\s+", " ", regex=True).str.strip()  # 공백/탭 정리
df["일자"] = pd.to_datetime(df["일자"], errors="coerce", format="mixed")  # 혼합 포맷 변환
before_drop = len(df)  # 드랍 전
df = df.dropna(subset=["일자"]).copy()  # 변환 실패 제거
print(f"2) 일자 파싱 완료 (드랍: {before_drop - len(df)}행, 남음: {len(df)}행)")  # 진행 출력

df["월"] = df["일자"].dt.to_period("M").astype(str)  # 월 생성

def normalize_item_code(code: str) -> str:  # 품목코드 표준화
    s = str(code).strip()
    if re.fullmatch(r"\d+", s):  # 숫자만
        s = s.lstrip("0")
        return s if s != "" else "0"
    return s

df["품목코드"] = df["품목코드"].astype(str).apply(normalize_item_code)  # 코드 정리
submit["품목코드"] = submit["품목코드"].astype(str).apply(normalize_item_code)  # 코드 정리

df["입고수량"] = pd.to_numeric(df["입고수량"], errors="coerce").fillna(0).astype(float)  # 입고수량 정리
df["출고수량"] = pd.to_numeric(df["출고수량"], errors="coerce").fillna(0).astype(float)  # 출고수량 정리
df["출고단가"] = pd.to_numeric(df["출고단가"], errors="coerce").fillna(0).astype(float)  # 출고단가 정리
df["재고수량"] = pd.to_numeric(df["재고수량"], errors="coerce").fillna(0).astype(float)  # 재고수량 정리

submit = submit.rename(columns={"본사\n실-전": "본사실전"})  # 컬럼명 통일
submit["본사실전"] = pd.to_numeric(submit["본사실전"], errors="coerce").fillna(0)  # 실-전 숫자화
submit["사용여부"] = submit["사용여부"].astype(str).str.strip()  # 과세/면세 정리

# 거래처명 [조정] 제외
before_adj = len(df)  # 제외 전
df = df[df["거래처명"].astype(str).str.strip() != "[조정]"].copy()  # [조정] 제거
print(f"3) [조정] 거래처 제외 완료 (제거: {before_adj - len(df)}행, 남음: {len(df)}행)")  # 진행 출력

df = df.sort_values(["일자"]).reset_index(drop=True)  # 정렬

use_map = submit.set_index("품목코드")["사용여부"].to_dict()  # 과세/면세 맵
surplus_pool = submit[submit["본사실전"] > 0][["품목코드", "본사실전", "사용여부"]].copy()  # 실-전 양수만
surplus_pool["남은여유"] = surplus_pool["본사실전"].astype(float)  # 여유량
print(f"4) 후보풀 생성 완료 (실-전 양수 품목수: {len(surplus_pool)})")  # 진행 출력

def get_use_flag(item_code: str) -> str:  # 과세/면세 조회
    return str(use_map.get(str(item_code).strip(), "")).strip()  # 반환

def fail_now(msg: str, context: dict | None = None) -> None:  # 실패 즉시 중단
    if context:
        print("  실패:", msg, "|", context)  # 원인/위치 출력
    else:
        print("  실패:", msg)  # 원인/위치 출력
    raise RuntimeError(msg)  # 즉시 중단

def build_item_meta_map(d: pd.DataFrame) -> dict:  # 품목명/규격/단위 대표값 생성
    meta = {}  # 결과
    for code, g in d.groupby("품목코드"):  # 품목별
        name = g["품목명"].dropna().astype(str)  # 품목명 후보
        spec = g["규격"].dropna().astype(str)  # 규격 후보
        unit = g["단위"].dropna().astype(str)  # 단위 후보
        if len(name) == 0 or len(spec) == 0 or len(unit) == 0:  # 하나라도 없으면
            continue  # 스킵
        meta[code] = {  # 최빈값 저장
            "품목명": name.mode().iloc[0],  # 최빈값
            "규격": spec.mode().iloc[0],  # 최빈값
            "단위": unit.mode().iloc[0],  # 최빈값
        }
    return meta  # 반환

print("5) 품목 메타 생성 시작")  # 진행 출력
df_meta = df[df["거래처명"].astype(str).str.strip() != "[조정]"].copy()  # 메타용(안전) [조정] 제외
item_meta = build_item_meta_map(df_meta)  # 메타 생성
print(f"5) 품목 메타 생성 완료 (메타 보유 품목수: {len(item_meta)})")  # 진행 출력

def apply_item_meta(row: pd.Series, item_code: str) -> pd.Series:  # 메타 반영
    row = row.copy()  # SettingWithCopyWarning 방지
    info = item_meta.get(str(item_code).strip(), None)  # 조회
    if info is None:  # 없으면
        return row  # 그대로
    row["품목명"] = info["품목명"]  # 교체
    row["규격"] = info["규격"]  # 교체
    row["단위"] = info["단위"]  # 교체
    return row  # 반환

def recalc_stock_for_items(d: pd.DataFrame, item_codes: list[str]) -> None:  # 품목별 재고수량 재계산
    if not item_codes:
        return
    item_set = set([str(x).strip() for x in item_codes])  # 대상 품목
    mask = d["품목코드"].astype(str).str.strip().isin(item_set)
    if not mask.any():
        return
    sub = d.loc[mask, ["품목코드", "일자", "입고수량", "출고수량"]].copy()
    sub["입고수량"] = pd.to_numeric(sub["입고수량"], errors="coerce").fillna(0).astype(float)
    sub["출고수량"] = pd.to_numeric(sub["출고수량"], errors="coerce").fillna(0).astype(float)
    sub = sub.sort_values(["품목코드", "일자"])  # 날짜순
    sub["재고수량"] = (sub["입고수량"] - sub["출고수량"]).groupby(sub["품목코드"]).cumsum()
    d.loc[sub.index, "재고수량"] = sub["재고수량"].values  # 결과 반영

month_price = (  # 월대표단가
    df[(df["출고수량"] > 0) & (df["거래처명"].astype(str).str.strip() != "[조정]")]  # [조정] 제외
    .groupby(["월", "품목코드"])["출고단가"]  # 월/품목
    .median()  # 중앙값
    .reset_index()  # 표로
    .rename(columns={"출고단가": "월대표단가"})  # 컬럼명
)

def pick_candidates(month: str, target_price: float, must_use_flag: str, exclude_item: str) -> pd.DataFrame:  # 후보 선택
    cand = surplus_pool[surplus_pool["남은여유"] > 0].copy()  # 여유만
    cand = cand[cand["사용여부"].astype(str).str.strip() == must_use_flag].copy()  # 과세/면세 일치
    cand = cand[cand["품목코드"].astype(str).str.strip() != str(exclude_item).strip()].copy()  # 동일 품목 제외
    cand["월"] = month  # 월 부여
    cand = cand.merge(month_price, on=["월", "품목코드"], how="left")  # 단가 붙이기
    cand["월대표단가"] = cand["월대표단가"].fillna(target_price)  # 없으면 대체
    cand["단가차이"] = (cand["월대표단가"] - target_price).abs()  # 차이
    cand = cand.sort_values(["단가차이", "남은여유"], ascending=[True, False])  # 정렬
    return cand  # 반환

history = []  # 이력

neg0 = df[df["재고수량"] < 0]  # 최초 음수(중간마이너스) 확인
print(f"6) 최초 중간마이너스(재고수량<0) 건수: {len(neg0)}")  # 진행 출력
if len(neg0) > 0:  # 있으면
    r0 = neg0.iloc[0]  # 첫 건
    print(f"   - 첫 음수: 일자={r0['일자']}, 품목={r0['품목코드']}, 재고수량={r0['재고수량']}")  # 디버깅 출력

# 샘플 설정이 로드되지 않은 경우를 대비한 기본값
SAMPLE_MODE = locals().get("SAMPLE_MODE", False)
SAMPLE_NEGATIVE_LIMIT = locals().get("SAMPLE_NEGATIVE_LIMIT", 5)
SAMPLE_MAX_MOVES = locals().get("SAMPLE_MAX_MOVES", 50)
SAMPLE_MAX_LOOPS = locals().get("SAMPLE_MAX_LOOPS", MAX_LOOP)

move_cnt = 0  # 이동 횟수
prev_signature = None  # 반복 감지용
repeat_count = 0  # 반복 횟수
sample_stop = False  # 샘플 중단 플래그

effective_max_loop = SAMPLE_MAX_LOOPS if SAMPLE_MODE else MAX_LOOP  # loop 제한

loop = 0  # 카운터
while loop < effective_max_loop:  # 반복
    loop += 1  # 증가

    neg_rows = df[df["재고수량"] < 0]  # 음수 탐지(재고수량 기준)
    if SAMPLE_MODE:
        neg_rows = neg_rows.head(SAMPLE_NEGATIVE_LIMIT)  # 샘플용 제한
    if neg_rows.empty:  # 없으면
        print(f"7) 완료: loop={loop-1}에서 중간마이너스 0건")  # 진행 출력
        break  # 종료

    neg_idx = int(neg_rows.index[0])  # 첫 음수 인덱스
    neg_row = df.loc[neg_idx]  # 행
    problem_item = str(neg_row["품목코드"]).strip()  # 문제코드
    problem_date = neg_row["일자"]  # 일자
    problem_month = str(neg_row["월"])  # 월
    problem_price = float(neg_row["출고단가"])  # 단가
    deficit = float(-neg_row["재고수량"])  # 부족(재고수량 기준)

    signature = (problem_item, problem_date, deficit)  # 반복 감지 키
    if signature == prev_signature:
        repeat_count += 1
    else:
        repeat_count = 0
    prev_signature = signature

    if repeat_count >= 1:  # 같은 문제행이 반복되면 중단
        msg = "같은 문제행 반복(재고수량 재계산 없음)"  # 사유
        history.append({"유형": "실패", "사유": msg, "문제품목": problem_item, "일자": problem_date, "부족수량": deficit})  # 기록
        fail_now(msg, {"단계": "반복 감지", "문제품목": problem_item, "일자": problem_date})  # 즉시 중단

    problem_use = get_use_flag(problem_item)  # 과세/면세
    print(f"[loop {loop}] 문제품목={problem_item}, 발생일={problem_date.date()}, 월={problem_month}, 부족={deficit:.0f}, 사용여부={problem_use}")  # 진행 출력

    if problem_use == "":  # 없으면
        msg = "문제품목 사용여부 없음(제출용)"  # 사유
        history.append({"유형": "실패", "사유": msg, "문제품목": problem_item, "일자": problem_date, "부족수량": deficit})  # 기록
        fail_now(msg, {"단계": "사용여부 확인", "문제품목": problem_item})  # 즉시 중단

    scope = df[df["재고수량"] < 0].copy()  # 범위: 재고수량 음수 행 전부

    if scope.empty:  # 없으면
        msg = "재고수량 음수 행 없음"  # 사유
        history.append({"유형": "실패", "사유": msg, "문제품목": problem_item, "일자": problem_date, "부족수량": deficit})  # 기록
        fail_now(msg, {"단계": "범위 추출", "문제품목": problem_item})  # 즉시 중단

    scope = scope.sort_values(["일자"], ascending=False)  # 최근부터

    candidates = pick_candidates(problem_month, problem_price, problem_use, problem_item)  # 후보
    print(f"  후보수={len(candidates)} (단가 기준 정렬)")  # 진행 출력

    if candidates.empty:  # 없으면
        msg = "과세/면세 일치 후보 없음"  # 사유
        history.append({"유형": "실패", "사유": msg, "문제품목": problem_item, "일자": problem_date, "부족수량": deficit, "문제사용여부": problem_use})  # 기록
        fail_now(msg, {"단계": "후보 선택", "문제품목": problem_item, "사용여부": problem_use})  # 즉시 중단

    moved_in_loop = False  # 이번 loop에서 변경 발생 여부
    resolved_problem = False  # 문제품목 해결 여부

    for _, src in scope.iterrows():  # 출고행 반복
        if deficit <= 0:  # 해결되면
            break  # 종료

        src_idx = int(src.name)  # 인덱스
        src_item = str(df.loc[src_idx, "품목코드"]).strip()  # 원품목
        if src_item != problem_item:  # 문제품목 행만 처리
            continue  # 다음
        movable = float(df.loc[src_idx, "출고수량"])  # 이동가능
        if movable <= 0:  # 없으면
            continue  # 다음

        for _, cand in candidates.iterrows():  # 후보 반복
            if deficit <= 0:  # 해결되면
                break  # 종료

            sub_item = str(cand["품목코드"]).strip()  # 대체코드
            remain = float(cand["남은여유"])  # 여유
            if remain <= 0:  # 없으면
                continue  # 다음

            q = min(deficit, movable, remain)  # 이동수량(부분대체 허용)
            if q <= 0:
                continue  # 다음 후보

            before_row = df.loc[src_idx].copy()  # 변경 전 기록
            is_partial = q < movable  # 부분대체 여부

            if is_partial:
                # 원행은 문제품목 그대로 유지하고 수량만 줄임
                df.loc[src_idx, "출고수량"] = movable - q  # 잔여 수량
                df.loc[src_idx, "출고금액"] = (movable - q) * float(before_row["출고단가"])  # 금액 재계산
                df.loc[src_idx, "재고수량"] = np.nan  # 재계산 예정
                df.loc[src_idx, "적요"] = f"부분대체(대체:{q})"  # 적요

                # 대체분은 새 행으로 추가
                new_row = before_row.copy()
                new_row["품목코드"] = sub_item  # 교체
                new_row["출고수량"] = q  # 수량
                new_row["출고단가"] = problem_price  # 단가 고정
                new_row["출고금액"] = q * problem_price  # 금액
                new_row["입고수량"] = 0.0  # 입고 0
                if "입고단가" in new_row.index:  # 체크
                    new_row["입고단가"] = 0.0  # 0
                if "입고금액" in new_row.index:  # 체크
                    new_row["입고금액"] = 0.0  # 0
                new_row["재고수량"] = np.nan  # 재계산 예정
                new_row["적요"] = f"대체출고(원:{before_row['품목코드']},수량:{q})"  # 적요
                new_row = apply_item_meta(new_row, sub_item)  # 메타 교체
                df = pd.concat([df, new_row.to_frame().T], ignore_index=True)  # 추가
            else:
                # 전체 대체는 기존 행을 교체
                df.loc[src_idx, "품목코드"] = sub_item  # 교체
                df.loc[src_idx, "출고수량"] = q  # 수량 유지(전체 대체)
                df.loc[src_idx, "출고단가"] = problem_price  # 단가 고정
                df.loc[src_idx, "출고금액"] = q * problem_price  # 금액 재계산
                df.loc[src_idx, "입고수량"] = 0.0  # 입고 0
                if "입고단가" in df.columns:  # 체크
                    df.loc[src_idx, "입고단가"] = 0.0  # 0
                if "입고금액" in df.columns:  # 체크
                    df.loc[src_idx, "입고금액"] = 0.0  # 0
                df.loc[src_idx, "재고수량"] = np.nan  # 재계산 예정
                df.loc[src_idx, "적요"] = f"대체출고(원:{before_row['품목코드']},수량:{q})"  # 적요
                df.loc[src_idx] = apply_item_meta(df.loc[src_idx], sub_item)  # 메타 교체

            surplus_pool.loc[surplus_pool["품목코드"] == sub_item, "남은여유"] -= q  # 여유 차감

            history.append({
                "유형": "부분대체" if is_partial else "대체",
                "월": problem_month,
                "일자": src["일자"],
                "원품목": before_row["품목코드"],
                "대체품목": sub_item,
                "이동수량": q,
                "원행잔여수량": movable - q,
                "문제사용여부": problem_use,
                "대체사용여부": get_use_flag(sub_item),
                "원행인덱스": src_idx,
                "변경전_출고수량": before_row["출고수량"],
                "변경후_출고수량": movable - q if is_partial else q,
                "변경전_출고단가": before_row["출고단가"],
                "변경후_출고단가": before_row["출고단가"] if is_partial else problem_price,
                "변경전_출고금액": before_row["출고금액"],
                "변경후_출고금액": (movable - q) * float(before_row["출고단가"]) if is_partial else q * problem_price,
                "대체_출고수량": q,
                "대체_출고단가": problem_price,
                "대체_출고금액": q * problem_price,
                "변경전_재고수량": before_row["재고수량"],
                "변경후_재고수량": np.nan,
            })  # 기록

            recalc_stock_for_items(df, [problem_item, sub_item])  # 문제/대체 품목만 재계산
            problem_neg = df[
                (df["품목코드"].astype(str).str.strip() == problem_item)
                & (df["재고수량"] < 0)
            ]
            if problem_neg.empty:
                deficit = 0  # 문제품목 음수 해결
                resolved_problem = True
            else:
                deficit = float(-problem_neg["재고수량"].min())  # 남은 부족 재계산

            move_cnt += 1  # 이동 카운트
            moved_in_loop = True  # 변경 표시

            if SAMPLE_MODE and move_cnt >= SAMPLE_MAX_MOVES:
                sample_stop = True  # 샘플 중단

            if move_cnt % PRINT_EVERY_N_MOVES == 0:  # 간헐 출력
                print(f"    이동 {move_cnt}건째: 대체품목={sub_item}, 이동={q:.0f}, 남은부족={deficit:.0f}")  # 진행 출력

            if resolved_problem:
                break  # 문제품목 해결되면 다음 loop로

            break  # 한 행을 대체했으면 다음 출고행

        if resolved_problem:
            break  # 문제품목 해결되면 출고행 반복 종료

    if SAMPLE_MODE and sample_stop:
        print("7-0) 샘플 이동 제한 도달, 중단")  # 샘플 중단 출력
        break  # 샘플 중단

    if deficit > 0 and not moved_in_loop:  # 변경 없음
        msg = "대체 가능한 출고행 없음"  # 사유
        history.append({"유형": "실패", "사유": msg, "문제품목": problem_item, "일자": problem_date, "부족수량": deficit})  # 기록
        fail_now(msg, {"단계": "대체 수행", "문제품목": problem_item})  # 즉시 중단

    df = df.sort_values(["일자"]).reset_index(drop=True)  # 정렬
    df["일자"] = pd.to_datetime(df["일자"], errors="coerce", format="mixed")  # 타입 보정
    df["월"] = df["일자"].dt.to_period("M").astype(str)  # 월 갱신

verify = df[df["재고수량"] < 0][["일자", "월", "품목코드", "재고수량", "입고수량", "출고수량", "출고단가", "창고명", "거래처명"]].copy()  # 검증(재고수량 기준)
hist_df = pd.DataFrame(history)  # 이력표

if SAVE_SAMPLE_ONLY:
    print("변경이력 샘플")  # 변경 이력 출력
    print(hist_df.head(SAMPLE_ROWS))  # 샘플 이력
else:
    print("변경이력 상세")  # 변경 이력 출력
    print(hist_df)  # 전체 이력

if SAVE_SAMPLE_ONLY:
    print("8-0) 샘플 저장 시작")  # 진행 출력
    with pd.ExcelWriter(OUT_PATH_SAMPLE, engine="openpyxl") as w:  # 샘플 저장
        df.head(SAMPLE_ROWS).to_excel(w, index=False, sheet_name="수불부_샘플")  # 조정본 샘플
        hist_df.head(SAMPLE_ROWS).to_excel(w, index=False, sheet_name="변경이력_샘플")  # 이력 샘플
        verify.head(SAMPLE_ROWS).to_excel(w, index=False, sheet_name="검증_샘플")  # 검증 샘플
    print("8-0) 샘플 저장 완료")  # 진행 출력
    print(OUT_PATH_SAMPLE)  # 샘플 경로
else:
    print("8) 결과 저장 시작")  # 진행 출력
    with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:  # 저장
        df.to_excel(w, index=False, sheet_name="수불부_조정")  # 조정본
        hist_df.to_excel(w, index=False, sheet_name="변경이력")  # 이력
        verify.to_excel(w, index=False, sheet_name="검증_중간마이너스")  # 검증
    print("8) 결과 저장 완료")  # 진행 출력
    print(OUT_PATH)  # 출력 경로

print(f"총 이동건수: {move_cnt}")  # 요약
print(f"최종 중간마이너스(재고수량<0) 건수: {len(verify)}")  # 요약
print(f"총 소요시간(초): {time.time() - t0:.1f}")  # 요약

2) 일자 파싱 시작
2) 일자 파싱 완료 (드랍: 0행, 남음: 509336행)
3) [조정] 거래처 제외 완료 (제거: 0행, 남음: 509336행)
4) 후보풀 생성 완료 (실-전 양수 품목수: 402)
5) 품목 메타 생성 시작
5) 품목 메타 생성 완료 (메타 보유 품목수: 1525)
6) 최초 중간마이너스(재고수량<0) 건수: 31782
   - 첫 음수: 일자=2025-01-01 00:00:00, 품목=4450, 재고수량=-2.0
[loop 1] 문제품목=4450, 발생일=2025-01-01, 월=2025-01, 부족=2, 사용여부=면세
  후보수=90 (단가 기준 정렬)
[loop 2] 문제품목=1415, 발생일=2025-01-01, 월=2025-01, 부족=2, 사용여부=면세
  후보수=90 (단가 기준 정렬)
    이동 20건째: 대체품목=4450, 이동=2, 남은부족=76
[loop 3] 문제품목=4450, 발생일=2025-01-01, 월=2025-01, 부족=2, 사용여부=면세
  후보수=89 (단가 기준 정렬)
    이동 40건째: 대체품목=4803, 이동=1, 남은부족=76
7-0) 샘플 이동 제한 도달, 중단
변경이력 샘플
      유형        월         일자   원품목  대체품목  이동수량  원행잔여수량 문제사용여부 대체사용여부   원행인덱스  \
0     대체  2025-01 2025-07-23  4450  1415   2.0     0.0     면세     면세  283477   
1     대체  2025-01 2025-07-18  4450  1415   1.0     0.0     면세     면세  277043   
2     대체  2025-01 2025-07-18  4450  1415   3.0     0.0     면세     면세  277844   
3     대체  2025-01 2025-07-15  4450  1415   1.0     0.0     면세     면세  271097   
4 

8-0) 샘플 저장 시작
8-0) 샘플 저장 완료
./output_조정본_샘플.xlsx
총 이동건수: 55
최종 중간마이너스(재고수량<0) 건수: 31817
총 소요시간(초): 700.1
